# 11 — Memory transformations: the L1 optimizers

Notebook 10 ended on a measurement and a cliffhanger. The L1 footprint
simulator, run on GPT-2, reported a forward peak of **6816 B** and a
forward+backward *joint* peak of **16400 B** — the backward pass keeps forward
activations alive to consume them, so the gradient program peaks **2.41×**
higher. That ratio *is* the reason L1 has more than a measuring stick. This
notebook is the three transforms that push the number back down, each one the
same loop: **measure → transform → re-measure**, with the reference layer's
`peak_memory` (bytes) and `ops_count` (FLOPs) as the two dials on the trade.

1. **Requested-gradients DCE** — *don't save what no gradient needs.* "Which
   gradients do I care about" turns out to be plain backward reachability on
   the joint DAG. No special cases, no "frozen layer" mode: freezing a
   parameter is dropping its gradient from the keep-set, and the
   saved-activation set shrinks by itself.
2. **Min-cut activation checkpointing** — *recompute instead of store.* The
   saved-for-backward set is a minimum cut between the primal inputs and the
   backward's consumers, with edge capacities in *exact bytes*. Three facts our
   representation knows — closed forms are free, views are uncuttable,
   contractions are recompute-banned — sharpen the classic AOTAutograd
   partitioner into something that reads its cost model straight off the IR.
3. **Segmented (checkpointed) fold adjoints** — *the same idea, inside a
   loop.* A `fold`'s backward can keep only segment-boundary states and
   recompute each segment's trajectory just-in-time. Sweeping the segment count
   K over a T=12 time-stepper reproduces Chen's √T heuristic from our own
   primitives — a memory/recompute curve with a measured minimum.

Everything here is a program-to-program rewrite: the input is IR, the output is
IR, and `grad`'s returned gradient-variable names survive untouched, so the
measurements are apples-to-apples.

In [1]:
import nbhelp  # noqa: F401  — puts tensorlib on sys.path
import numpy as np
from tensorlib import Tensor
from tensorlib.autodiff import grad
from tensorlib.ir import Instr, Program, run
from tensorlib.memory import peak_memory
from tensorlib.opcount import ops_count
from tensorlib.transforms import checkpoint, dce
from tensorlib.zoo import gpt2

In [2]:
def I(var, op, operands=(), **params):
    return Instr(var, op, tuple(operands), params)


def T(arr, names):
    return Tensor.from_numpy(np.asarray(arr, dtype=np.float64), names)


def loss_head(m):
    # Append a scalar training loss (sum of squared outputs) so `grad` seeds
    # itself with 1 — the ordinary "one number to differentiate" endpoint. It
    # replaces notebook 10's external `dL` VJP seed with a computed reduction;
    # the forward peak is unchanged (the extra ops are a pointwise + a reduce).
    return Program(
        m.program.instrs
        + (
            I("zsq", "pointwise", (m.out, m.out), f="mul"),
            I("zloss", "reduce", ("zsq",), f="sum", dims=("t", "v")),
        )
    )

## Move 1 — Requested-gradients DCE

Build the GPT-2 joint program the ordinary way: a scalar training loss, then
`grad`. The joint carries a gradient for **every** input — the token embedding
`x`, the positions, and all 28 weight tensors of the two transformer blocks.
That is why it is big.

In [3]:
m = gpt2()
lp = loss_head(m)
jp, grads = grad(lp, "zloss", m.inputs)

fwd_peak = peak_memory(lp, m.inputs).peak_bytes
joint_peak = peak_memory(jp, m.inputs).peak_bytes
print(f"forward loss     : {len(lp.instrs):>4} instrs   peak {fwd_peak:>6} B")
print(f"forward+backward : {len(jp.instrs):>4} instrs   peak {joint_peak:>6} B   "
      f"({joint_peak / fwd_peak:.2f}x)")
print(f"the joint computes gradients for all {len(m.inputs)} inputs "
      f"(x, positions, and every weight)")

forward loss     :  220 instrs   peak   6816 B
forward+backward :  767 instrs   peak  16240 B   (2.38x)
the joint computes gradients for all 29 inputs (x, positions, and every weight)


Now the DCE thesis in one line. Suppose the only gradient you actually want is
`dL/dx` — you are, say, doing input attribution, or every weight is frozen.
`dce(prog, keep)` walks the instructions *backwards* from the kept set and
drops any instruction that does not transitively feed something kept. The
weight-gradient contractions (`activationᵀ · upstream`, the expensive part)
never reach `dL/dx`, so they vanish — and with them the activations they were
the only consumers of.

In [4]:
keep_x = dce(jp, (grads["x"], "zloss"))
peak_x = peak_memory(keep_x, m.inputs).peak_bytes
print(f"request only dL/dx : {len(keep_x.instrs):>4} instrs   peak {peak_x:>6} B")
print(f"  dropped {len(jp.instrs) - len(keep_x.instrs)} backward instrs; "
      f"{joint_peak - peak_x} B off the peak")

full = run(jp, m.inputs)
only = run(keep_x, m.inputs)
same = np.allclose(
    only[grads["x"]].to_numpy(order=("t", "d")),
    full[grads["x"]].to_numpy(order=("t", "d")),
    rtol=1e-12,
)
print("  dL/dx is bit-for-bit identical to the full joint:", same)

request only dL/dx :  654 instrs   peak  12016 B
  dropped 113 backward instrs; 4224 B off the peak
  dL/dx is bit-for-bit identical to the full joint: True


**Freezing is not a mode — it is a keep-set.** GPT-2 here is two blocks
(`L0`, `L1`) plus the embedding and LM head. "Freeze block `L0`" means: request
every gradient *except* the `L0.*` weight gradients. It is the same
reachability query with a different keep-set, and the peak falls smoothly as
you request less. The extremes are "train everything" (all 29 gradients) and
"only `dL/dx`" (freeze all weights) — no special case sits between them.

In [5]:
def keepset(freeze=()):
    kept = tuple(grads[n] for n in m.inputs
                 if not any(n.startswith(p) for p in freeze))
    return dce(jp, kept + ("zloss",))


rows = [
    ("train everything      ", jp),
    ("freeze block L0        ", keepset(freeze=("L0.",))),
    ("only dL/dx (freeze all)", keep_x),
]
for label, prog in rows:
    pk = peak_memory(prog, m.inputs).peak_bytes
    print(f"{label}: {len(prog.instrs):>4} instrs   peak {pk:>6} B")

train everything      :  767 instrs   peak  16240 B
freeze block L0        :  710 instrs   peak  14640 B
only dL/dx (freeze all):  654 instrs   peak  12016 B


## Move 2 — Min-cut activation checkpointing

DCE removes work you never wanted. Checkpointing trades work you *do* want —
storage for recomputation. The joint program splits at the loss into forward
and backward; the backward reads a set **R** of forward values, and naively all
of R stays live across the boundary. Instead we pick a **saved** set S and
recompute everything else in R from `S ∪ inputs`, placed lazily just before
first use so the recomputed values do not all coexist.

Choosing S is the **min-cut formulation** (REPRESENTATIONS.md's prior-art map:
this is AOTAutograd's partitioner) — a minimum cut between *sources* (primal
inputs + recompute-banned outputs) and a *sink* fed by R, with node capacities
= exact bytes from the layout shadows. Start with the smallest interesting
chain: square, exponentiate, contract, square-the-scalar.

In [6]:
def chain():
    # x -> sq (mul) -> e (exp) -> r (reduce, a contraction) -> loss = r*r
    prog = Program(
        (
            I("x", "input"),
            I("sq", "pointwise", ["x", "x"], f="mul"),        # 64 B
            I("e", "pointwise", ["sq"], f="exp"),             # 64 B
            I("r", "reduce", ["e"], f="sum", dims=("i",)),    # 8 B — the contraction
            I("loss", "pointwise", ["r", "r"], f="mul"),      # scalar
        )
    )
    return prog, {"x": T(0.1 * np.arange(8.0), ("i",))}


cprog, cin = chain()
cjp, cg = grad(cprog, "loss", cin)
ck = checkpoint(cjp, "loss", cin)
print("saved      :", list(ck.saved))
print("recomputed :", ck.recomputed)
print(f"boundary   : {ck.bytes_before} B naively  ->  {ck.bytes_after} B saved")

saved      : [('r', 8)]
recomputed : ('sq', 'e')
boundary   : 72 B naively  ->  8 B saved


The min cut saves exactly `r` — 8 bytes. Naively the boundary would hold the
big pointwise `sq` (64 B) *and* `r` (72 B total); instead `sq` and `e` are
recomputed from the free-to-keep input `x`, and only the banned reduce output
`r` is stored. Recomputation in pure SSA is duplication under fresh `.rc`
names: `sq.rc` and `sq` are **two variables, one semantic value** — the
`name ≠ value` price REPRESENTATIONS.md predicted, harmless for run/measure.

In [7]:
print("the rewritten backward duplicates the cheap chain under fresh .rc names:")
for ins in ck.program.instrs:
    if ".rc" in ins.var:
        print(f"    {ins.var:<7}= {ins.op}{tuple(ins.operands)}  {ins.params}")

gj = run(cjp, cin)[cg["x"]].to_numpy()
gc = run(ck.program, cin)[cg["x"]].to_numpy()
print("\ndL/dx identical (sq and sq.rc are one value under two names):",
      np.allclose(gj, gc, rtol=1e-12))

the rewritten backward duplicates the cheap chain under fresh .rc names:
    sq.rc  = pointwise('x', 'x')  {'f': 'mul'}
    e.rc   = pointwise('sq.rc',)  {'f': 'exp'}

dL/dx identical (sq and sq.rc are one value under two names): True


**The three representation sharpenings.** The classic partitioner treats every
tensor as an opaque blob with one size. Ours reads three structural facts off
the IR, which is what makes the cut land where it does:

- **Closed forms cost 0 — free to "save".** `iota`, `const`, and
  masks-held-as-guards are computed from position, not stored, so their cut
  capacity is zero. They never enter the budget.
- **Views are uncuttable.** A slice/shift/repeat is an alias with no bytes of
  its own; saving it would pin its root. So a view gets **∞** capacity at the
  view — the cut must fall on its *root* (or recompute the view, which is
  free).
- **Contractions are recompute-banned = fresh demand sources.** `reduce`,
  `scan`, `fold` are the operations whose recompute would double real FLOPs, so
  by default they *source* demand into the cut: the saved set must land
  at-or-after them. Pointwise chains and layout ops recompute freely — the
  fusion-cheap region.

The ban is a **policy dial**, not a cost model. Set `ban=()` and even the
reduce is recomputed from the input — the boundary drops to zero saved bytes,
pure rematerialization, gradients still identical.

In [8]:
ck0 = checkpoint(cjp, "loss", cin, ban=())
print(f"ban=() (recompute all): saved {ck0.bytes_after} B, "
      f"recomputed {ck0.recomputed}")
g0 = run(ck0.program, cin)[cg["x"]].to_numpy()
print("gradient still identical:", np.allclose(gj, g0, rtol=1e-12))

ban=() (recompute all): saved 0 B, recomputed ('sq', 'e', 'r')
gradient still identical: True


### GPT-2, with real numbers

Now the same cut on the GPT-2 joint. The boundary shrinks by more than half,
and because recompute is placed just-in-time, the peak follows it down.

In [9]:
ck_g = checkpoint(jp, "zloss", m.inputs)
pk_before = peak_memory(jp, m.inputs).peak_bytes
pk_after = peak_memory(ck_g.program, m.inputs).peak_bytes
print(f"saved boundary : {ck_g.bytes_before} B  ->  {ck_g.bytes_after} B   "
      f"({ck_g.bytes_after / ck_g.bytes_before:.0%})")
print(f"joint peak     : {pk_before} B  ->  {pk_after} B   "
      f"({pk_after / pk_before:.0%})")
print()
print("the cut lands on the contraction outputs — exactly what theory predicts:")
for v, b in ck_g.saved[:9]:
    print(f"    {v:<7}{b:>5} B")
print(f"    ...  {len(ck_g.saved)} saved tensors in all")

saved boundary : 8448 B  ->  3936 B   (47%)
joint peak     : 16240 B  ->  12368 B   (76%)

the cut lands on the contraction outputs — exactly what theory predicts:
    mu        32 B
    var       32 B
    q        192 B
    k        192 B
    v        192 B
    sc       256 B
    mx        64 B
    z         64 B
    ctx      192 B
    ...  27 saved tensors in all


Read the saved names: `mu`/`var` (LayerNorm reduce statistics), `q`/`k`/`v`
(the QKV projections — matmul is repeat·mul·**reduce**), `sc` (attention
scores, a contraction), `ctx` (the context, attn·V), `m1` (the MLP hidden
contraction), `logits`. The cut saved the outputs of the banned reductions and
recomputed all the cheap pointwise/view glue between them. Gradients are
unchanged — check the input embedding and a weight deep in block 0.

In [10]:
ej = run(jp, m.inputs)
ec = run(ck_g.program, m.inputs)
for v in ("x", "L0.wq"):
    ok = np.allclose(
        ec[grads[v]].to_numpy(order=m.inputs[v].names),
        ej[grads[v]].to_numpy(order=m.inputs[v].names),
        rtol=1e-10,
    )
    print(f"grad {v:<7} identical: {ok}")

grad x       identical: True
grad L0.wq   identical: True


**The passes compose.** DCE prunes what you never asked for; checkpointing
plans what remains. Order them — prune first, then cut — and the wins stack.
Each pass alone lands the peak near 75% of the joint; together they reach 64%.

In [11]:
ck_dce = checkpoint(keep_x, "zloss", m.inputs)   # keep_x = the DCE'd (only dL/dx) program
pk_dce = peak_memory(ck_dce.program, m.inputs).peak_bytes
print(f"DCE alone        : peak {peak_x:>6} B   ({peak_x / joint_peak:.0%} of joint)")
print(f"checkpoint alone : peak {pk_after:>6} B   ({pk_after / joint_peak:.0%} of joint)")
print(f"DCE + checkpoint : peak {pk_dce:>6} B   ({pk_dce / joint_peak:.0%} of joint)")

DCE alone        : peak  12016 B   (74% of joint)
checkpoint alone : peak  12368 B   (76% of joint)
DCE + checkpoint : peak  10352 B   (64% of joint)


## Move 3 — Segmented (checkpointed) fold adjoints

The min cut above works on a straight-line DAG. A `fold` (notebook 08) is a
*loop* whose body is itself a program, and its store-everything adjoint keeps
all T step-trajectories alive to feed the reverse pass — the loop analogue of
the saved-activations problem. `grad(..., fold_segments=K)` applies **Chen-style
uniform checkpointing** *inside* the fold adjoint: cut the time axis into K
equal segments, keep only the K segment-**boundary** states up front, and
recompute each segment's trajectory just-in-time during its own backward sweep.
So ~T/K + K states live instead of T.

Take the 1D FDTD leapfrog time-stepper from notebook 08 (two field states,
E and H; the step is a 17-instruction Program) and run it for T=12 steps.

In [12]:
# 1D FDTD leapfrog: state = (E on 6 nodes, H on 5 edges); no per-step elements.
N, C = 6, 0.3
FDTD_STEP = Program(
    (
        I("E", "input"),
        I("H", "input"),
        I("Es", "shift", ["E"], deltas={"x": -1}),
        I("Ea", "slice", ["Es"], ranges={"x": (0, N - 1)}),
        I("Eb", "slice", ["E"], ranges={"x": (0, N - 1)}),
        I("dE", "pointwise", ["Ea", "Eb"], f="sub"),
        I("cH", "const", [], value=C, dims=(("x", (0, N - 1)),)),
        I("dHs", "pointwise", ["cH", "dE"], f="mul"),
        I("H1", "pointwise", ["H", "dHs"], f="add"),
        I("Hs", "shift", ["H1"], deltas={"x": 1}),
        I("Ha", "slice", ["H1"], ranges={"x": (1, N - 1)}),
        I("Hb", "slice", ["Hs"], ranges={"x": (1, N - 1)}),
        I("dH", "pointwise", ["Ha", "Hb"], f="sub"),
        I("pdH", "pad", ["dH"], fill=0.0, extents={"x": (0, N)}),
        I("cE", "const", [], value=C, dims=(("x", (0, N)),)),
        I("dEs", "pointwise", ["cE", "pdH"], f="mul"),
        I("E1", "pointwise", ["E", "dEs"], f="add"),
    )
)


def fdtd_prog(steps):
    return Program(
        (
            I("E0", "input"),
            I("H0", "input"),
            I("Ef", "fold", ["E0", "H0"], step=FDTD_STEP, dim="t",
              state=("E", "H"), element=(), carry={"E": "E1", "H": "H1"},
              out=("emit", "E1"), extent=(0, steps)),
            I("E2", "pointwise", ["Ef", "Ef"], f="mul"),
            I("loss", "reduce", ["E2"], f="sum", dims=("t", "x")),
        )
    )


E = np.zeros(N); E[2] = 1.0          # a pulse
fin = {"E0": T(E, ("x",)), "H0": T(np.zeros(N - 1), ("x",))}
prog = fdtd_prog(12)
print("FDTD, T=12 steps; loss = sum of squared E over the whole trajectory")

FDTD, T=12 steps; loss = sum of squared E over the whole trajectory


Sweep K over the divisors of 12. `peak_memory` gives the bytes, `ops_count`
the recompute cost; the gradients must be identical to the store-everything
adjoint (K=None) at every K.

In [13]:
print(f"{'K':>4}  {'segments':>12}  {'peak B':>7}  {'ops (wtd)':>10}  {'dL/d(E0,H0)':>12}")
base = None
for K in (None, 2, 3, 4, 6, 12):
    jpf, gf = grad(prog, "loss", fin, fold_segments=K)
    env = run(jpf, fin)
    res = {v: env[gf[v]].to_numpy() for v in ("E0", "H0")}
    peak = peak_memory(jpf, fin).peak_bytes
    ops = ops_count(jpf, fin).weighted()
    if base is None:
        base, match = res, "baseline"
    else:
        ok = all(np.allclose(res[v], base[v], rtol=1e-9) for v in ("E0", "H0"))
        match = "identical" if ok else "DIFFERS"
    seg = "store all" if K is None else f"{K} x len {12 // K}"
    print(f"{str(K):>4}  {seg:>12}  {peak:>7}  {ops:>10.0f}  {match:>12}")

   K      segments   peak B   ops (wtd)   dL/d(E0,H0)
None     store all     2680        5507      baseline
   2     2 x len 6     1824        5879     identical
   3     3 x len 4     1816        6003     identical


   4     4 x len 3     1816        6065     identical


   6     6 x len 2     1816        6127     identical
  12    12 x len 1     2176        6189     identical


There is the curve. Peak falls **2680 → 1816 B** as segmentation kicks in, then
*rises* again at K=12 (2176 B) where the per-segment boundary-state bookkeeping
outweighs the trajectory it saves. The minimum sits at K=3–6, and √12 ≈ 3.46 —
**Chen's √T heuristic, reproduced from our own primitives** rather than
asserted. The `ops (wtd)` column is the price: recompute cost climbs
monotonically with K. And the gradients are identical at every K — the segments
change *when* work happens, never *what* it computes.

The segment count must divide the fold extent (K = a global uniform stride);
an indivisible K is refused loudly, not silently rounded.

In [14]:
try:
    grad(prog, "loss", fin, fold_segments=5)  # 12 % 5 != 0
except ValueError as e:
    print("refused:", e)

refused: fold_segments=5 must divide the fold extent 12 (pad the dim or pick a divisor)


**What remains** (CONCERNS.md #27, kept honest). This is *uniform* (Chen-style)
checkpointing, not the optimal **binomial revolve** (Griewank & Walther,
O(log T) memory) — that is the next refinement. K is **global per `grad` call**,
not per-fold. K must **divide** T (pad the dim or pick a divisor first). And
one caveat that spans both Move 2 and Move 3: min-cut checkpointing minimizes
the *boundary*, and the peak usually follows (GPT-2: 76%) because recompute is
placed just-in-time — but no theorem connects the two. Direct-peak minimization
is schedule search / pebbling, a later pass. The ban set is a policy, not a
cost model (a tiny reduce is banned while a huge pointwise recomputes), and the
`.rc` duplicates break `name = value` until a value-numbering pass sees through
them.

---

## The L1 loop, and why it is the template

Every move here was the same three beats: **measure** with `peak_memory` /
`ops_count`, **transform** the IR (DCE prune, min-cut rewrite, segmented
adjoint), **re-measure** to price the trade. Nothing was estimated — the layout
algebra is the exact alias theory, so the bytes are computed on the nose, and
the transforms are program-to-program rewrites whose gradients stay bit-for-bit
identical.

That loop is not special to L1. LEVELS.md lays the ladder above it: **L2**
storage (which bytes — liveness coloring, in-place), **L3** placement (where —
mesh-labeled dims, collectives from alignment diagnosis), **L4** kernels (who —
tiling, red-blue pebbling, the flash-attention derivation), **L5** schedule
(when — the timeline simulator). Each rung swaps in its own cost model and its
own transforms, but the shape is identical: *a white-box representation makes
the measurement exact, and an IR-to-IR rewrite pays down the number the
measurement exposed.* L1 is where that loop first closes; the rest of the ladder
just turns the same crank on a bigger machine.